# Prepare and read an entangled state

Goal: prepare two correlated qubits and read their named outcomes. A register holds a quantum state, not a sampled integer. `Output` lets the function allocate the register. `prepare_bell_state` expresses the preparation; the compiler chooses its gates. Bell label 1 means (|00> - |11>)/sqrt(2). Run from the repository root with the project environment.

In [ ]:
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QNum, prepare_bell_state, qfunc


@qfunc
def bell(pair: Output[QNum]) -> None:
    prepare_bell_state(1, pair)


## Build and execute

These are separate steps: synthesis builds a reusable program, and `fq.run` performs local execution. The bounds limit resources; exceeding a bound must fail, not silently change the problem.

In [ ]:
program = synth.synthesize(
    bell, max_qubits=2, max_workspace=0, memory_limit_bytes=1024**2
)
result = fq.run(program)
outcomes = dict(program.probabilities(result, registers=("pair",)))
print(outcomes)
print(program.report["resources"])


## Interpret the result

The named integer outcomes 0 and 3 encode 00 and 11. Each has probability one half. These are exact simulator probabilities, not finite-shot counts. Correlated probabilities alone do not prove entanglement: the acceptance test also checks complex amplitudes and relative phase.

Try Bell label 2 and compare its outcomes and statevector. No wire allocation or bitstring decoder is needed.

## Request actual finite-shot counts

Use FlagQuantum's existing Runtime count request on the synthesized Circuit IR.
The selected qubits come from the named logical port; no physical wire numbers
are written. The output label `pair` selects its measurement result. Returned
bitstrings follow that port's declared bit order (high bit first).

This creates a separate measured execution plan for the same circuit. A sealed
synthesis Program currently refuses output or shot overrides, and its exact
named probability reader is not a count decoder. Keep these boundaries explicit
until a named finite-shot contract is admitted. These local samples model ideal
terminal measurements; they are not hardware shots or statevector amplitudes.


In [ ]:
shots = 4096
sampled = fq.run(
    program.circuit,
    outputs=fq.counts(qubits=program.registers["pair"], name="pair"),
    options=fq.ExecutionOptions(shots=shots, seed=37),
)
shot_counts = sampled.measurement("pair").value[0]
print("Actual counts:", shot_counts)
print(
    "Empirical frequencies:",
    {word: count / shots for word, count in shot_counts.items()},
)
print(
    "Shot uncertainty:",
    sampled.measurement("pair").statistics["outcome_standard_error"],
)
assert sum(shot_counts.values()) == shots
assert set(shot_counts) <= {"00", "11"}


Repeat with seed 91. Counts may change while exact probabilities remain one half.
Changing the seed tests reproducibility; statistical agreement does not prove the Bell relative phase.
